# NYC Taxi Trip Duration Prediction using PySpark ML

This project predicts NYC taxi trip duration using Random Forest Regression in Apache Spark. The workflow includes exploratory data analysis, feature engineering, data cleaning, model training, and evaluation.

## Load Dataset

The dataset is loaded from Databricks Volume into a Spark DataFrame.

In [0]:
df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/workspace/default/nyc/train.csv")
)

## Exploratory Data Analysis

In this section, we inspect the dataset, check for missing values, analyze summary statistics, and identify potential outliers.

Minimum and Maximum Trip Duration

In [0]:
from pyspark.sql import functions as F

df.select(
            (F.min("trip_duration")/3600).alias("min_trip_duration"),
            (F.max("trip_duration")/3600).alias("max_trip_duration")
    ).show()


+--------------------+-----------------+
|   min_trip_duration|max_trip_duration|
+--------------------+-----------------+
|2.777777777777778E-4|979.5227777777778|
+--------------------+-----------------+



%md
Detect the Outliers

In [0]:
from pyspark.sql import functions as F

df.orderBy(F.desc("trip_duration"))\
    .select("id","pickup_datetime","dropoff_datetime","trip_duration")\
        .limit(10).show()

+---------+-------------------+-------------------+-------------+
|       id|    pickup_datetime|   dropoff_datetime|trip_duration|
+---------+-------------------+-------------------+-------------+
|id0053347|2016-02-13 22:46:52|2016-03-25 18:18:14|      3526282|
|id1325766|2016-01-05 06:14:15|2016-01-31 01:01:07|      2227612|
|id0369307|2016-02-13 22:38:00|2016-03-08 15:57:38|      2049578|
|id1864733|2016-01-05 00:19:42|2016-01-27 11:08:38|      1939736|
|id1942836|2016-02-15 23:18:06|2016-02-16 23:17:58|        86392|
|id0593332|2016-05-31 13:00:39|2016-06-01 13:00:30|        86391|
|id0953667|2016-05-06 00:00:10|2016-05-07 00:00:00|        86390|
|id2837671|2016-06-30 16:37:52|2016-07-01 16:37:39|        86387|
|id1358458|2016-06-23 16:01:45|2016-06-24 16:01:30|        86385|
|id2589925|2016-05-17 22:22:56|2016-05-18 22:22:35|        86379|
+---------+-------------------+-------------------+-------------+



In [0]:
from pyspark.sql import functions as F

df.select(
    F.expr("percentile(trip_duration, array(0.25, 0.5, 0.75, 0.95, 0.99))")
      .alias("Percentiles")
).show(truncate=False)

+--------------------------------------+
|Percentiles                           |
+--------------------------------------+
|[397.0, 662.0, 1075.0, 2104.0, 3440.0]|
+--------------------------------------+



In [0]:
df.filter(F.col("trip_duration") > 3440) \
  .select("id", "pickup_datetime", "dropoff_datetime", "trip_duration") \
  .show(20, truncate=False)

+---------+-------------------+-------------------+-------------+
|id       |pickup_datetime    |dropoff_datetime   |trip_duration|
+---------+-------------------+-------------------+-------------+
|id3827863|2016-04-19 11:29:08|2016-04-19 12:27:56|3528         |
|id3402983|2016-06-30 15:48:06|2016-06-30 17:31:13|6187         |
|id2693863|2016-03-18 08:22:10|2016-03-18 09:47:19|5109         |
|id3307903|2016-02-20 04:03:06|2016-02-21 03:33:00|84594        |
|id3607196|2016-01-26 11:22:27|2016-01-26 12:20:57|3510         |
|id2029339|2016-01-22 14:13:46|2016-01-22 15:15:21|3695         |
|id0631822|2016-05-17 14:17:48|2016-05-17 15:26:06|4098         |
|id3893063|2016-06-02 17:32:41|2016-06-02 18:42:43|4202         |
|id1091477|2016-05-07 18:36:22|2016-05-08 18:32:11|86149        |
|id1040844|2016-06-03 15:10:40|2016-06-03 16:36:32|5152         |
|id2553024|2016-04-23 13:43:33|2016-04-23 14:47:19|3826         |
|id0896335|2016-01-25 11:07:10|2016-01-25 12:04:58|3468         |
|id2295021

## Feature Engineering

To improve model performance, I created additional features:

- Pickup Hour
- Weekend Indicator
- Rush Hour Indicator
- Haversine Trip Distance

Extract pick up hour

In [0]:
from pyspark.sql import functions as F

df=df.withColumn("pick_hour",F.hour("pickup_datetime"))

Create new feature is_rush_hour

In [0]:
from pyspark.sql import functions as F
df=df.withColumn("pick_hour",F.hour("pickup_datetime"))
df = df.withColumn(
    "is_rush_hour",
    F.when(
        (
            ((F.col("pick_hour") >= 8) & (F.col("pick_hour") <= 10)) |
            ((F.col("pick_hour") >= 16) & (F.col("pick_hour") <= 18))
        ),
        1
    ).otherwise(0)
)

In [0]:
from pyspark.sql import functions as F
df=df.withColumn("pick_hour",F.hour("pickup_datetime"))
df = df.withColumn(
    "is_weekend",
    F.when(
        
            (F.dayofweek("pickup_datetime").isin(1,7))
        ,
        1
    ).otherwise(0)
)
df.show(10)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+---------+------------+----------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|pick_hour|is_rush_hour|is_weekend|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+---------+------------+----------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|       17|           1|         0|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-

In [0]:
df = df.withColumn(
    "pickup_lat_rad",
    F.radians("pickup_latitude")
)

df = df.withColumn(
    "pickup_lon_rad",
    F.radians("pickup_longitude")
)

df = df.withColumn(
    "dropoff_lat_rad",
    F.radians("dropoff_latitude")
)

df = df.withColumn(
    "dropoff_lon_rad",
    F.radians("dropoff_longitude")
)

In [0]:
df = df.withColumn(
    "dlat",
    F.col("dropoff_lat_rad") - F.col("pickup_lat_rad")
)

df = df.withColumn(
    "dlon",
    F.col("dropoff_lon_rad") - F.col("pickup_lon_rad")
)

In [0]:
R = 6371.0

In [0]:
a = (
    F.pow(F.sin(F.col("dlat") / 2), 2)
    +
    F.cos(F.col("pickup_lat_rad"))
    * F.cos(F.col("dropoff_lat_rad"))
    * F.pow(F.sin(F.col("dlon") / 2), 2)
)

c = 2 * F.asin(F.sqrt(a))

df = df.withColumn(
    "trip_distance_km",
    R * c
)

## Model Training

The cleaned dataset is split into training and testing sets. A Random Forest Regressor is trained using Spark ML Pipeline.

In [0]:
df_clean = df.filter(
    (F.col("trip_distance_km") <= 50) &
    (F.col("trip_duration") <= 7200)
)

In [0]:
train_df, test_df = df_clean.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "vendor_encoded",
        "passenger_count",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "pick_hour",
        "is_weekend",
        "is_rush_hour",
        "trip_distance_km"
    ],
    outputCol="features"
)

In [0]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=[
        vendor_indexer,
        encoder,
        assembler,
        rf
    ]
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5029287016806893>, line 5
      1 from pyspark.ml import Pipeline
      3 pipeline = Pipeline(
      4     stages=[
----> 5         vendor_indexer,
      6         encoder,
      7         assembler,
      8         rf
      9     ]
     10 )

NameError: name 'vendor_indexer' is not defined

In [0]:
model_clean = pipeline.fit(train_df)

In [0]:
predictions_clean = model_clean.transform(test_df)

## Model Evaluation

The trained model is evaluated using RMSE, MAE, and R² metrics.

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_clean = RegressionEvaluator(
    labelCol="trip_duration",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(predictions_clean)

mae_clean = RegressionEvaluator(
    labelCol="trip_duration",
    predictionCol="prediction",
    metricName="mae"
).evaluate(predictions_clean)

r2_clean = RegressionEvaluator(
    labelCol="trip_duration",
    predictionCol="prediction",
    metricName="r2"
).evaluate(predictions_clean)

print("RMSE:", rmse_clean)
print("MAE :", mae_clean)
print("R2  :", r2_clean)

In [0]:
rf_model_clean = model_clean.stages[-1]

importances = rf_model_clean.featureImportances

print(importances)

In [0]:
assembler.getInputCols()

In [0]:
feature_vector = predictions_clean.select("features").first()[0]

print(feature_vector)
print("Number of features:", feature_vector.size)